In [ ]:
# import required libraries
import requests
import pandas as pd
from bs4 import BeautifulSoup
import pyperclip
import os
import re
import json

In [ ]:
# read in the `exploration` html directly from a URL and prettify it using BeautifulSoup, then write it to a file
link = input("Enter the URL of the site you want to scrape: ")

# get the HTML content of the page
response = requests.get(link)
soup = BeautifulSoup(response.content, 'html.parser')

# filter to only the section of interest, which is the table containing the data we want to extract
target_table = soup.find("div", class_="full_column non-widget-area") # IMPORTANT: this is the div that contains the table we want to extract

# strip out unnecessary scripts and styles
for script in target_table(["script", "style"]):
    script.decompose()

# apply formatting to the HTML using BeautifulSoup's prettify method
pretty_html = target_table.prettify()

# write the prettified HTML to a file
if pretty_html:
    # pyperclip.copy(pretty_html)
    # create a regular expression pattern to match the url and extract the name of the page
    # pat = r"https?://(?:www\.)?([^/]+)"
    pat = re.compile(r"\b[^.\n]+\.([^.\n/]+)\.")
    org = re.search(pat, link).group(1)
    # write the prettified HTML to a file in the current working directory
    with open(f"{org}.html", "w", encoding="utf-8") as f:
        f.write(pretty_html)

In [ ]:
# find all blocks where the class has the pattern r"eventItem
pat = re.compile(r"^eventItem.*")

# find all divs with the class that matches the pattern
event_items = target_table.find_all(class_=pat)

# create a dictionary to hold the event data
event_dict = {}
df = pd.DataFrame()

# parse each event to get name, link to details, date (range), description, image link, location, price, and ticket link (if available)
for event in event_items:
    # print(event)
    # get the name of the event
    try:
        event_dict["name"] = event.find("h3").get_text(strip=True)
    except AttributeError:
        event_dict["name"] = None

    # get the link to the details page
    try:
        event_dict["details_link"] = event.find("a")["href"]
    except AttributeError:
        event_dict["details_link"] = None

    # get the date range of the event
    try:
        event_dict["date_range"] = event.find("div", class_="date").get("aria-label").strip()
    except AttributeError:
        event_dict["date_range"] = None

    # get the description of the event
    try:
        event_dict["description"] = event.find("h4").get_text(strip=True)
    except AttributeError:
        event_dict["description"] = None

    # get the image link of the event
    try:
        event_dict["image_link"] = event.find("img")["src"]
    except AttributeError:
        event_dict["image_link"] = None
        
    # get the ticket link of the event (if available)
    try:
        event_dict["ticket_link"] = event.find("a", class_="tickets onsalenow").get("href").strip()
    except AttributeError:
        event_dict["ticket_link"] = None

    # if the ticket link is available, get the price range of the event
    if event_dict["ticket_link"]:
        try:
            # get the ticket page HTML
            ticket_response = requests.get(event_dict["ticket_link"])
            ticket_soup = BeautifulSoup(ticket_response.content, 'html.parser')

            # get the section of interest, which is the div containing the price range
            ticket_section = ticket_soup.find("ul", id="list-view") # IMPORTANT: this is the div that contains the price range we want to extract
            for ticket in ticket_section:
                # use list comprehension to get the text of each li element and store it in a list
                ticket_prices = [li.get_text(strip=True) for li in ticket.find_all("li", role="menuitem")]

            # DEV: add ticket_prices to the event_dict
            event_dict["ticket_prices"] = ticket_prices
        except AttributeError:
            ticket_prices = None

    # concatenate the event_dict to the dataframe
    df = pd.concat([df, pd.DataFrame([event_dict])], ignore_index=True)

In [ ]:
# follow the link to the details page
